# MML Chapter 2: Linear Algebra

> **Premise:** Vectors, matrices, and linear maps are the algebraic backbone of ML — from solving systems of equations to representing PCA, neural network layers, and least-squares regression.

**Source:** Deisenroth, Faisal & Ong — *Mathematics for Machine Learning*, Ch 2 (pp 23–75)

---

## Contents
1. Vectors as arrays, dot products, matrix multiplication
2. Solving $Ax = b$: three outcomes
3. Row echelon form: manual Gaussian elimination
4. Null space and column space
5. Basis and rank
6. Linear maps: transforming a unit circle
7. Exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import null_space

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 100

---
## 1. Vectors as Arrays, Dot Products, Matrix Multiplication

A **vector** is anything you can add together and scale. In NumPy, vectors live as 1-D arrays, and matrices as 2-D arrays.

**Dot product** (standard inner product on $\mathbb{R}^n$):
$$\mathbf{x} \cdot \mathbf{y} = \mathbf{x}^\top \mathbf{y} = \sum_{i=1}^n x_i y_i$$

**Matrix multiplication** — the $(i,j)$ entry of $AB$ is the dot product of row $i$ of $A$ with column $j$ of $B$:
$$(AB)_{ij} = \sum_k A_{ik} B_{kj}$$

Key rules:
- $AB \neq BA$ in general (not commutative)
- $(AB)C = A(BC)$ (associative)
- Dimensions must be compatible: $(n \times k)(k \times m) = (n \times m)$
- $(AB)^\top = B^\top A^\top$ (transpose reverses order)

In [ ]:
# --- Vectors ---
x = np.array([1.0, 2.0, 3.0])
y = np.array([4.0, 5.0, 6.0])

dot_product = np.dot(x, y)          # or x @ y
print(f"x         = {x}")
print(f"y         = {y}")
print(f"x · y     = {dot_product}   (= 1*4 + 2*5 + 3*6)")

# --- Matrices ---
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])   # shape 3×2

B = np.array([[7, 8, 9],
              [10, 11, 12]])  # shape 2×3

AB = A @ B   # (3×2)(2×3) → 3×3
BA = B @ A   # (2×3)(3×2) → 2×2

print(f"\nA (3×2):\n{A}")
print(f"\nB (2×3):\n{B}")
print(f"\nAB (3×3):\n{AB}")
print(f"\nBA (2×2):\n{BA}")
print(f"\nAB ≠ BA: shapes differ ({AB.shape} vs {BA.shape})")

# Verify (AB)^T == B^T A^T
lhs = AB.T
rhs = B.T @ A.T
print(f"\n(AB)^T == B^T A^T: {np.allclose(lhs, rhs)}")

---
## 2. Solving $Ax = b$: Three Outcomes

Given a system $Ax = b$, exactly one of three things happens:

| Case | Geometry | Conditions |
|------|----------|------------|
| **Unique solution** | Lines/planes cross at one point | $A$ is square and invertible ($\det A \neq 0$) |
| **No solution** | Lines/planes are parallel | $b \notin \text{col}(A)$; system is inconsistent |
| **Infinitely many solutions** | Lines/planes overlap | $A$ has a non-trivial null space; free variables exist |

No fourth option exists — this trichotomy is fundamental.

In [ ]:
# === Case 1: Unique solution ===
A1 = np.array([[2.0, 1.0],
               [1.0, 3.0]])
b1 = np.array([5.0, 10.0])

x1 = np.linalg.solve(A1, b1)
print("Case 1 — Unique solution:")
print(f"  A:\n{A1}")
print(f"  b = {b1}")
print(f"  x = {x1}")
print(f"  Verify Ax = b: {np.allclose(A1 @ x1, b1)}")
print(f"  det(A) = {np.linalg.det(A1):.4f}  (nonzero → unique)")

# === Case 2: No solution ===
# Two parallel lines: x + y = 1  and  x + y = 3
A2 = np.array([[1.0, 1.0],
               [1.0, 1.0]])
b2 = np.array([1.0, 3.0])

print("\nCase 2 — No solution (inconsistent):")
print(f"  A:\n{A2}")
print(f"  b = {b2}")
try:
    x2 = np.linalg.solve(A2, b2)
    print(f"  x = {x2}")
except np.linalg.LinAlgError as e:
    print(f"  LinAlgError: {e}")
print(f"  det(A) = {np.linalg.det(A2):.4f}  (zero → not invertible)")
print(f"  rank(A) = {np.linalg.matrix_rank(A2)},  rank([A|b]) = {np.linalg.matrix_rank(np.column_stack([A2, b2]))}")
print("  rank([A|b]) > rank(A) → no solution")

# === Case 3: Infinitely many solutions ===
# Two coincident lines: 2x + y = 4  and  4x + 2y = 8
A3 = np.array([[2.0, 1.0],
               [4.0, 2.0]])
b3 = np.array([4.0, 8.0])

print("\nCase 3 — Infinitely many solutions:")
print(f"  A:\n{A3}")
print(f"  b = {b3}")
print(f"  det(A) = {np.linalg.det(A3):.4f}  (zero → not invertible)")
print(f"  rank(A) = {np.linalg.matrix_rank(A3)},  rank([A|b]) = {np.linalg.matrix_rank(np.column_stack([A3, b3]))}")
print("  rank([A|b]) == rank(A) < n → infinitely many solutions")
# Particular + null-space solution
x_part = np.linalg.lstsq(A3, b3, rcond=None)[0]
ns = null_space(A3)
print(f"  Particular solution x_p = {x_part}")
print(f"  Null space direction   = {ns.flatten()}")
print(f"  General solution: x_p + t * null_space_direction, t ∈ ℝ")

---
## 3. Row Echelon Form: Manual Gaussian Elimination

**Gaussian elimination** transforms a matrix to **row echelon form (REF)** using three elementary row operations:
1. Swap two rows
2. Multiply a row by a nonzero scalar
3. Add a multiple of one row to another

**Row echelon form** has a staircase pattern — each pivot is to the right of the pivot in the row above, and all entries below a pivot are zero.

**Reduced REF (RREF)** additionally has all pivots equal to 1, and all other entries in pivot columns are zero.

The general solution = (particular solution) + (null space).

In [ ]:
def gaussian_elimination(A_in, b_in, verbose=True):
    """Manual Gaussian elimination to row echelon form, then back-substitution."""
    n = len(b_in)
    # Augmented matrix [A | b]
    M = np.hstack([A_in.astype(float), b_in.reshape(-1, 1).astype(float)])

    if verbose:
        print("Augmented [A|b]:")
        print(M)

    for col in range(n):
        # Find pivot row (partial pivoting)
        max_row = col + np.argmax(np.abs(M[col:, col]))
        if M[max_row, col] == 0:
            continue
        # Swap rows
        if max_row != col:
            M[[col, max_row]] = M[[max_row, col]]
            if verbose:
                print(f"\nSwap row {col} ↔ row {max_row}:")
                print(M)
        # Eliminate below
        for row in range(col + 1, n):
            factor = M[row, col] / M[col, col]
            if factor != 0:
                M[row] -= factor * M[col]
                if verbose:
                    print(f"\nR{row} ← R{row} - ({factor:.4f}) * R{col}:")
                    print(M)

    if verbose:
        print("\n→ Row echelon form reached.")

    # Back substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        if M[i, i] == 0:
            x[i] = 0  # free variable (set to 0 for particular solution)
        else:
            x[i] = (M[i, -1] - np.dot(M[i, i+1:n], x[i+1:n])) / M[i, i]
    return x, M


A = np.array([[2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

print("=== Gaussian Elimination on a 3×3 System ===")
x_sol, M_ref = gaussian_elimination(A, b, verbose=True)
print(f"\nSolution: x = {x_sol}")
print(f"Verify Ax = b: {np.allclose(A @ x_sol, b)}")

# Cross-check with numpy
x_numpy = np.linalg.solve(A, b)
print(f"NumPy solve: x = {x_numpy}")

---
## 4. Null Space and Column Space

For a matrix $A$ (shape $m \times n$), two fundamental subspaces are:

**Null space (kernel):** $\ker(A) = \{\mathbf{x} \in \mathbb{R}^n : A\mathbf{x} = \mathbf{0}\}$
- Captures all vectors that $A$ maps to zero — the "information lost" by the transformation.

**Column space (image/range):** $\text{col}(A) = \{A\mathbf{x} : \mathbf{x} \in \mathbb{R}^n\}$
- Span of the columns of $A$ — all reachable outputs.

**Rank-nullity theorem:**
$$\text{rank}(A) + \dim(\ker(A)) = n$$

The rank (dimension of column space) and nullity (dimension of null space) always sum to the number of columns.

In [ ]:
from scipy.linalg import null_space

# A matrix with rank 2 (one-dimensional null space)
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

print(f"A (3×3):\n{A}\n")

# --- Null space ---
ns = null_space(A)
print(f"Null space basis (scipy.linalg.null_space):")
print(ns)
print(f"dim(null space) = {ns.shape[1]}")
print(f"Verify A @ null_space ≈ 0: max abs = {np.max(np.abs(A @ ns)):.2e}\n")

# --- Column space ---
# Use QR decomposition to find an orthonormal basis for the column space
rank = np.linalg.matrix_rank(A)
Q, R = np.linalg.qr(A)
col_space_basis = Q[:, :rank]  # first 'rank' columns of Q

print(f"rank(A) = {rank}")
print(f"Column space basis (orthonormal, via QR):")
print(col_space_basis)

# --- Rank-nullity theorem ---
n_cols = A.shape[1]
nullity = ns.shape[1]
print(f"\nRank-nullity check:")
print(f"  rank(A) = {rank}")
print(f"  nullity  = {nullity}")
print(f"  n (cols) = {n_cols}")
print(f"  rank + nullity = {rank + nullity} = n = {n_cols}  ✓")

---
## 5. Basis and Rank

A **basis** of a vector space $V$ is a set of vectors that:
1. **Spans** $V$ — every vector in $V$ can be written as a linear combination
2. Is **linearly independent** — no vector is redundant

Every vector in $V$ has a **unique** representation in terms of the basis.

**Dimension** = number of basis vectors (not the size of the vectors).
- $\text{span}\{[0,1]^\top\}$ is 1-dimensional even though each vector has 2 components.

**Rank of a matrix** = number of linearly independent columns = number of linearly independent rows:
$$\text{rk}(A) = \text{rk}(A^\top)$$

A matrix $A$ ($m \times n$) has **full rank** when $\text{rk}(A) = \min(m, n)$.

In [ ]:
# Several matrices to compare ranks
examples = {
    "Full rank (3×3, invertible)": np.array([[1, 0, 0],
                                              [0, 2, 0],
                                              [0, 0, 3]], dtype=float),
    "Rank 2 (3×3, one zero eigenvalue)": np.array([[1, 2, 3],
                                                     [4, 5, 6],
                                                     [7, 8, 9]], dtype=float),
    "Rank 1 (outer product)": np.outer([1, 2, 3], [4, 5, 6]).astype(float),
    "Full rank (2×3, wide)": np.array([[1, 0, 2],
                                        [0, 1, 3]], dtype=float),
    "Rank deficient (2×3)": np.array([[1, 2, 3],
                                       [2, 4, 6]], dtype=float),
}

for label, M in examples.items():
    r = np.linalg.matrix_rank(M)
    m, n = M.shape
    full = (r == min(m, n))
    print(f"{label}")
    print(f"  shape={M.shape}, rank={r}, full_rank={full}")
    print()

# Standard basis of R^3
print("Standard basis of ℝ³:")
I3 = np.eye(3)
for i in range(3):
    print(f"  e{i+1} = {I3[i]}")

# Showing that a non-standard basis spans the same space
B = np.array([[1, 1, 0],
              [0, 1, 1],
              [1, 0, 1]], dtype=float)
print(f"\nAlternative basis B, rank = {np.linalg.matrix_rank(B)}  (still spans ℝ³)")
print(f"B:\n{B}")

---
## 6. Linear Maps: Transforming a Unit Circle

Every matrix represents a **linear map** (transformation). The key property:
$$\Phi(\lambda \mathbf{x} + \psi \mathbf{y}) = \lambda \Phi(\mathbf{x}) + \psi \Phi(\mathbf{y})$$

Grid lines stay parallel and evenly spaced — the signature of linearity.

Applying a $2 \times 2$ matrix $A$ to all points on the **unit circle** reveals what the map does geometrically:
- A **rotation matrix** keeps it a circle (distances preserved)
- A general matrix **stretches it into an ellipse** (different scaling in different directions)
- The axes of the ellipse align with the matrix's **singular vectors**, and the axis lengths are the **singular values** (more on this in Ch 4)

In [ ]:
# Unit circle: parametric
theta = np.linspace(0, 2 * np.pi, 300)
circle = np.array([np.cos(theta), np.sin(theta)])  # shape (2, 300)

# Three 2×2 matrices with different behaviors
matrices = [
    ("Identity\n$A = I$",
     np.eye(2)),
    ("Shear\n$A = [[1,1.5],[0,1]]$",
     np.array([[1.0, 1.5], [0.0, 1.0]])),
    ("Stretch + Rotate\n$A = [[2,0.5],[0.3,1.5]]$",
     np.array([[2.0, 0.5], [0.3, 1.5]])),
    ("Rotation 45°\n$R(\\pi/4)$",
     np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
               [np.sin(np.pi/4),  np.cos(np.pi/4)]])),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (title, A) in zip(axes, matrices):
    # Transform the unit circle
    ellipse = A @ circle

    ax.plot(circle[0], circle[1], 'b--', alpha=0.4, linewidth=1.5, label='Unit circle')
    ax.plot(ellipse[0], ellipse[1], 'r-', linewidth=2, label='Transformed')

    # Draw the image of standard basis vectors
    origin = np.zeros(2)
    for i, (col, color) in enumerate(zip(A.T, ['darkgreen', 'purple'])):
        ax.annotate('', xy=col, xytext=origin,
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))
        ax.text(col[0]*1.1, col[1]*1.1, f"$Ae_{i+1}$", color=color, fontsize=9)

    lim = max(2.5, np.abs(ellipse).max() * 1.2)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    sv = np.linalg.svd(A, compute_uv=False)
    ax.set_xlabel(f"σ₁={sv[0]:.2f}, σ₂={sv[1]:.2f}", fontsize=8)

fig.suptitle("Linear Maps: Unit Circle → Image (red = transformed, blue dashed = original)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Exercises

Work through these to solidify your understanding. Each involves hands-on coding.

**Exercise 1 — Matrix inverse via augmentation:**
Given $A = [[3, 1], [2, 4]]$, compute $A^{-1}$ manually by augmenting with $I$ and row-reducing to $[I \mid A^{-1}]$. Verify that $A \cdot A^{-1} = I$.

**Exercise 2 — Rank-nullity verification:**
Create a random $4 \times 6$ matrix, find its rank using `np.linalg.matrix_rank`, and find its null space using `scipy.linalg.null_space`. Verify that rank + nullity = 6.

**Exercise 3 — Composition of transformations:**
Define two $2 \times 2$ matrices $A$ (a shear) and $B$ (a $90°$ rotation). Apply them in both orders ($AB$ and $BA$) to the unit circle. Plot the results side by side and describe how they differ geometrically.

**Exercise 4 — Gaussian elimination on an under-determined system:**
Solve the system
$$\begin{cases} x + 2y + 3z = 6 \\ 2x + 5y + 2z = 4 \\ 3x + 8y + 0z = 0 \end{cases}$$
using `np.linalg.lstsq`. Is the system consistent? If so, how many free variables are there? Find two distinct solutions.